In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

output_dir = '/content/drive/MyDrive/omg_diploma_2025/Diploma_Data'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)


Mounted at /content/drive


In [ ]:
!pip install -q datasets pandas conllu tqdm

from datasets import load_dataset
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import json
import pandas as pd
import conllu
import requests
import numpy as np
import ast
import re
import random
import zipfile
import io
import gc


In [ ]:
# datasets ideas (for start)
# https://huggingface.co/datasets/IlyaGusev/ru_news
# https://huggingface.co/datasets/IlyaGusev/gazeta (https://arxiv.org/pdf/2006.11063)
# https://huggingface.co/datasets/RussianNLP/rucola (https://github.com/IlyaGusev/RuCoLA/tree/main) # for evaluation
# https://huggingface.co/datasets/Helsinki-NLP/opus_books
# https://huggingface.co/datasets/universal-dependencies/universal_dependencies (https://github.com/UniversalDependencies/UD_Russian-SynTagRus/tree/master)


# print("Loading Lenta.ru...")
# dataset = load_dataset("ilearning/lenta_ru_news", split="train", streaming=True)


### google drive setup

# Text cleaning

In [ ]:
MAX_WORDS_PER_CHUNK = 250
# TARGET_COUNT = 15000
WIKI_LIMIT = 200000

LABEL_MAP = {
    "O": 0, # Space only
    ".": 1,
    ",": 2,
    "!": 3,
    "?": 4,
    ":": 5,
    ";": 6,
    "-": 7,

    '"': 8,
    ' "': 9,
    '" ': 10,

    ', "': 11,
    ': "': 12,
    '. "': 13,

    '",': 14,
    '".': 15,
    '"?': 16,
    '"!': 17,
    '...': 18,

    '- "': 19,
    '", -': 20, # direct speech with closing "
    '!" -': 21,
    '?" -': 22,
    '. -': 23,

    '""': 24,      # Double close (поступил в учреждение "школа "Эврика"".)

    "! -": 25, # direct speech without closing "
    "? -": 26,
    ", -": 27,
}

ID_TO_LABEL = {v: k for k, v in LABEL_MAP.items()}


In [ ]:
for k, v in LABEL_MAP.items(): print(k, v)


O 0
. 1
, 2
! 3
? 4
: 5
; 6
- 7
" 8
 " 9
"  10
, " 11
: " 12
. " 13
", 14
". 15
"? 16
"! 17
... 18
- " 19
", - 20
!" - 21
?" - 22
. - 23
"" 24
! - 25
? - 26
, - 27


In [ ]:
# to do:
# 1. handle parentheses, brackets etc. (just return them)
# 2. minus before numbers etc (-8,5 'C)
# 3.

In [ ]:
file_path = os.path.join(output_dir, "label_map.json")
with open(file_path, 'w', encoding='utf-8') as f:
    json.dump(ID_TO_LABEL, f, ensure_ascii=False)


In [ ]:
def robust_clean_text(text):
    text = re.sub(r'[\u2010-\u2015]', '-', text)
    text = re.sub(r'[«»“”„]', '"', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [ ]:
# def get_smart_tokens(text):
#     """
#     Tokens that should NOT be split
#     1. URLs/Emails (dots/slashes)
#     2. Numbers (integers, decimals, negatives -8,5)
#     3. Words (including words with internal hyphens like 'кто-то')
#     4. Wrappers (parentheses)
#     """
#     token_pattern = r'''
#         (?x)                        # Verbose mode
#         (?:[({[\"\']*?)             # 1. Optional Prefix: ( { [ " '
#         (?:                         # 2. Main Body (One of these):
#             https?://\S+|www\.\S+   #    A. URL (catch this first!)
#             |
#             -?\d+(?:[.,/]\d+)+      #    B. Complex Num: -8.5, 1/2, 10.000 (No spaces!)
#             |
#             -?\d+                   #    C. Simple Integer: 5, -5 (Negative sign attached)
#             |
#             [\w]+(?:-[\w]+)*        #    D. Word: word, кто-то (Hyphens inside words allowed)
#         )
#         (?:[)\]}\"\']*?)            # 3. Optional Suffix: ) ] } " '
#     '''
#     return list(re.finditer(token_pattern, text))


In [ ]:
# def normalize_gap(gap_text):
#     """
#     Converts raw gap text into a label ID key.
#     """
#     if not gap_text.strip():
#         return "O"

#     # Remove generic whitespace
#     clean = gap_text.strip()

#     # --- HANDLING YOUR SPECIFIC "BUGGED" GAPS ---

#     # Logic: Remove spaces around marks to find the skeleton
#     skeleton = re.sub(r'\s+', '', clean)

#     # Heuristics for spacing (needed for quotes)
#     has_leading_space = gap_text.startswith(' ')

#     # 1. Simple Marks
#     if skeleton in ['.', ',', '!', '?', ':', ';', '-']:
#         return skeleton

#     # 2. Quotes (Context Aware)
#     if skeleton == '"':
#         return ' "' if has_leading_space else '" '

#     # 3. Complex Clusters (Skeleton matching)
#     mapping = {
#         ',"' : ', "',  ':"' : ': "',  '."' : '. "',
#         '",' : '",',   '".' : '".',   '"?' : '"?',   '"!' : '"!',
#         '-"' : '- "',  '",-' : '", -',
#         '!-' : '! -',  '?-' : '? -',  '.-' : '. -',  ',-' : ', -',
#         '""' : '""',   '"" ' : '"" ', '"",' : '"",'
#     }

#     return mapping.get(skeleton, "O") # Default to O if unknown mess


In [ ]:
# def extract_labels_standardized(text):
#     text = robust_clean_text(text)

#     # Use the Smart Tokenizer
#     matches = get_smart_tokens(text)

#     result_pairs = []

#     for i, match in enumerate(matches):
#         word = match.group() # The token (including brackets/negatives)
#         start_pos = match.end()

#         # Find the gap until the next token starts
#         if i + 1 < len(matches):
#             end_pos = matches[i+1].start()
#             gap_text = text[start_pos:end_pos]
#         else:
#             # End of sentence case (usually ignored or treated as final mark)
#             # For BERT training, we usually drop the very last mark or treat it as padding,
#             # but let's capture it.
#             gap_text = text[start_pos:]

#         # Determine Label
#         label_key = normalize_gap(gap_text)
#         label_id = LABEL_MAP.get(label_key, 0)

#         result_pairs.append({"word": word, "label": label_id})

#     return result_pairs


In [ ]:
def get_canonical_label(gap_text):
    if not gap_text.strip():
        return "O"

    # capture spacing before stripping to distinguishes ' "' from '" '
    has_leading_space = gap_text.startswith(' ')
    # assume standard punctuation implies trailing space for simple marks
    # but for quotes it's different
    skeleton = re.sub(r'\s+', '', gap_text)

    if skeleton == '.': return '.'
    if skeleton == ',': return ','
    if skeleton == '!': return '!'
    if skeleton == '?': return '?'
    if skeleton == ':': return ':'
    if skeleton == ';': return ';'
    if skeleton == '-': return '-'

    if skeleton == '"':
        # If leading space => Open Quote
        if has_leading_space:
            return ' "'
        # otherwise assume Close Quote
        return '" '

    if skeleton == ',"': return ', "'
    if skeleton == ':"': return ': "'
    if skeleton == '."': return '. "'

    if skeleton == '",': return '",'
    if skeleton == '".': return '".'
    if skeleton == '"?': return '"?'
    if skeleton == '"!': return '"!'

    if skeleton == '-"': return '- "'
    if skeleton == '",-': return '", -'
    if skeleton == '!"-': return '!"-'
    if skeleton == '?"-': return '?"-'

    if skeleton == '!-': return '! -'
    if skeleton == '?-': return '? -'
    if skeleton == '.-': return '. -'
    if skeleton == ',-': return ', -'

    if skeleton == '""': return '""' # Fallback
    if skeleton == '"",': return '"",'

    return "O"


In [ ]:

def extract_labels_standardized(text):
    text = robust_clean_text(text)

    # number:
    # -?              : Optional negative
    # \d+             : digit start
    # (?:[.,]\d+)+    : followed by a group of (dot OR comma) + digits, repeated 1+ times
    # punct doesn't divide parts of number
    number_pattern = r"-?\d+(?:[.,]\d+)+"

    # word:
    # -?           -> optional minus (in numbers like -1.5)
    # [(\[]?       -> Optional ( or [
    # [\w]+        -> word characters
    # (?:-[\w]+)*  -> Optional hyphen
    # [)\]]?       -> Optional closing ) or ]
    word_pattern = r"-?[(\[]?[\w]+(?:-[\w]+)*[)\]]?"

    # standalone brackets
    brackets_pattern = r"[()]|[\[\]]"

    full_pattern = f"{number_pattern}|{word_pattern}|{brackets_pattern}"

    word_iter = list(re.finditer(full_pattern, text))

    result_pairs = []

    for i, match in enumerate(word_iter):
        word = match.group()
        start_gap = match.end()

        if i + 1 < len(word_iter):
            end_gap = word_iter[i+1].start()
            raw_gap = text[start_gap:end_gap]
        else:
            raw_gap = text[start_gap:]

        label_str = get_canonical_label(raw_gap)
        label_id = LABEL_MAP.get(label_str, 0)

        result_pairs.append({"word": word, "label": label_id})

    return result_pairs

In [ ]:
def create_json_dataset(texts, output_filename):
    processed_data = []
    print(f"Processing {len(texts)} texts for {output_filename}...")

    for i, text in enumerate(tqdm(texts)):
        try:
            pairs = extract_labels_standardized(text)
        except:
            continue

        if len(pairs) < 5: continue # Skip tiny fragments

        tokens = [p['word'] for p in pairs]
        ner_tags = [p['label'] for p in pairs]

        processed_data.append({
            "tokens": tokens,
            "ner_tags": ner_tags
        })

    full_path = os.path.join(output_dir, output_filename)

    with open(full_path, 'w', encoding='utf-8') as f:
        json.dump(processed_data, f, ensure_ascii=False)

    del processed_data
    gc.collect()

    print(f"Saved {full_path}")

In [ ]:
def fix_wiki_encoding(text):
    if not isinstance(text, str):
        # If bytes, decode it
        try:
            text = text.decode('utf-8')
        except:
            pass
    if text.startswith("b'") or text.startswith('b"'):
        try:
            # literal_eval turns "b'\xd0\xb0'" (str) -> b'\xd0\xb0' (bytes)
            # then .decode('utf-8') turns bytes -> "а" (str)
            text = ast.literal_eval(text).decode('utf-8')
        except (ValueError, SyntaxError):
            pass

    tags_to_remove = [
        "_START_ARTICLE_",
        "_START_SECTION_",
        # "_START_PARAGRAPH_",
        # "_NEWLINE_"
    ]

    for tag in tags_to_remove:
        text = text.replace(tag, " ")

    text = re.sub(r'\s+', ' ', text)

    return text.strip()

# Syntagrus preparation

In [ ]:
# URLs for training
syntagrus_urls = [
    "https://raw.githubusercontent.com/UniversalDependencies/UD_Russian-SynTagRus/master/ru_syntagrus-ud-train-a.conllu",
    "https://raw.githubusercontent.com/UniversalDependencies/UD_Russian-SynTagRus/master/ru_syntagrus-ud-train-b.conllu",
    "https://raw.githubusercontent.com/UniversalDependencies/UD_Russian-SynTagRus/master/ru_syntagrus-ud-train-c.conllu"
]


In [ ]:
def process_syntagrus_files(urls, dataset_name):
    chunks = []
    print(f"--- PROCESSING {dataset_name} ---")
    for url in urls:
        fname = url.split("/")[-1]
        if not os.path.exists(fname):
            r = requests.get(url)
            with open(fname, "wb") as f: f.write(r.content)

        current_chunk = []
        current_word_count = 0
        current_doc_id = None
        current_genre = None

        with open(fname, "r", encoding="utf-8") as f:
            for sentence in tqdm(conllu.parse_incr(f)):
                sent_id = sentence.metadata['sent_id']
                text = sentence.metadata['text']
                genre = sentence.metadata.get('genre', 'unknown')
                doc_id = sent_id.rsplit("_", 1)[0] if "_" in sent_id else sent_id

                if doc_id != current_doc_id:
                    if current_chunk:
                        chunks.append({"text": " ".join(current_chunk), "genre": current_genre})
                    current_chunk = []
                    current_word_count = 0
                    current_doc_id = doc_id
                    current_genre = genre
                elif current_genre == 'unknown' and genre != 'unknown':
                    current_genre = genre

                sent_len = len(text.split())
                if current_chunk and (current_word_count + sent_len > MAX_WORDS_PER_CHUNK):
                    chunks.append({"text": " ".join(current_chunk), "genre": current_genre})
                    current_chunk = []
                    current_word_count = 0
                current_chunk.append(text)

        if current_chunk:
            chunks.append({"text": " ".join(current_chunk), "genre": current_genre})
    return pd.DataFrame(chunks)


In [ ]:
df_syn = process_syntagrus_files(syntagrus_urls, "SynTagRus TRAIN")


--- PROCESSING SynTagRus TRAIN ---


24516it [00:22, 1112.92it/s]
24299it [00:12, 1929.71it/s]
20816it [00:10, 2044.00it/s]


# Gazeta load

In [ ]:
ext_journalism = []
gazeta_url = "https://huggingface.co/datasets/IlyaGusev/gazeta/resolve/main/gazeta_train.jsonl"

response = requests.get(gazeta_url, stream=True)
total_size = int(response.headers.get('content-length', 0))
line_count = 0
# target_count = 15000

for line in tqdm(response.iter_lines(), desc="Gazeta Lines"): # , total=target_count
    # if line_count >= target_count:
    #     break
    if line:
        try:
            data = json.loads(line)
            # 'text' is the article body
            t = data['text'].replace('\n', ' ')
            if len(t) > 100:
                ext_journalism.append(t)
                line_count += 1
        except:
            continue

print(f"Loaded {len(ext_journalism)} Gazeta samples.")

Gazeta Lines: 1it [00:00, 3675.99it/s]

Loaded 0 Gazeta samples.


# Helsinki Opus books

In [ ]:
# from oxen import RemoteRepo
# repo = RemoteRepo("Helsinki-NLP/opus_books")
# repo.download("en-ru/opus_books_en-ru_train.parquet", revision="main")

# import pandas as pd
# df = pd.read_parquet("https://hub.oxen.ai/api/repos/Helsinki-NLP/opus_books/file/main/en-ru/opus_books_en-ru_train.parquet")
# print(df.head())

# "https://opus.nlpl.eu/Books/ru&en/v1/Books#download"



In [ ]:
# import nltk
# nltk.download('punkt')
from nltk.tokenize import sent_tokenize

def chunk_external_text(text, max_words=MAX_WORDS_PER_CHUNK):
    sentences = sent_tokenize(text, language='russian')
    chunks = []
    current_chunk_sentences = []
    current_word_count = 0

    for sentence in sentences:
        sentence_words = sentence.split()
        sentence_len = len(sentence_words)

        if current_chunk_sentences and (current_word_count + sentence_len > max_words):
            chunks.append(" ".join(current_chunk_sentences))
            current_chunk_sentences = [sentence]
            current_word_count = sentence_len
        else:
            current_chunk_sentences.append(sentence)
            current_word_count += sentence_len

    if current_chunk_sentences:
        chunks.append(" ".join(current_chunk_sentences))

    return chunks

In [ ]:
ext_fiction = []
opus_url = "https://object.pouta.csc.fi/OPUS-Books/v1/moses/en-ru.txt.zip"

response = requests.get(opus_url, stream=True)
total_size = int(response.headers.get('content-length', 0))

with io.BytesIO() as buffer:
    for chunk in response.iter_content(chunk_size=4096): buffer.write(chunk)
    print("Unzipping...")
    with zipfile.ZipFile(buffer) as z:
        with z.open('Books.en-ru.ru') as f:
            for line in tqdm(f, desc="Opus Stream"):
                try:
                    t = line.decode('utf-8').strip()
                    if len(t) > 20:
                        chunks = chunk_external_text(t)
                        ext_fiction.extend(chunks)
                except: continue
print(f"Loaded {len(ext_fiction)} Opus chunks.")


Unzipping...


Opus Stream: 17496it [00:13, 1310.12it/s]

Loaded 0 Opus chunks.


# Wikipedia

In [ ]:
def check_heading_punctuation(heading_text):
    if not heading_text:
        return False
    pairs = extract_labels_standardized(heading_text)
    if not pairs:
        return False

    last_word_label = pairs[-1]['label']
    return last_word_label != LABEL_MAP["O"]

In [ ]:
import re
import ast

def parse_wiki_segments(raw_item_text):
    text_content = ""
    if not isinstance(raw_item_text, str):
        try:
            text_content = raw_item_text.decode('utf-8')
        except (UnicodeDecodeError, AttributeError):
            try:
                text_content = ast.literal_eval(raw_item_text).decode('utf-8')
            except (ValueError, SyntaxError, AttributeError):
                return []
    else:
        text_content = raw_item_text

    text_content = text_content.replace('_NEWLINE_', ' ')
    text_content = re.sub(r'\s+', ' ', text_content).strip()

    tag_definitions = {
        "_START_ARTICLE_": "article_heading",
        "_START_SECTION_": "section_heading",
        "_START_PARAGRAPH_": "paragraph"
    }

    tag_pattern = '|'.join(re.escape(tag) for tag in tag_definitions.keys())

    segments = []
    last_end = 0

    first_match = re.search(tag_pattern, text_content)
    if first_match:
        pre_tag_text = text_content[0:first_match.start()]
        cleaned_pre_tag_text = robust_clean_text(pre_tag_text)
        if cleaned_pre_tag_text:
            segments.append({"text": cleaned_pre_tag_text, "type": "paragraph"})

    for match in re.finditer(tag_pattern, text_content):
        tag = match.group(0)
        start_index = match.end()

        next_match = re.search(tag_pattern, text_content[start_index:])
        if next_match:
            end_index = start_index + next_match.start()
        else:
            end_index = len(text_content)

        segment_text = text_content[start_index:end_index]
        cleaned_segment_text = robust_clean_text(segment_text)

        if cleaned_segment_text:
            segment_type = tag_definitions.get(tag, "unknown")

            if segment_type in ["article_heading", "section_heading"]:
                if not check_heading_punctuation(cleaned_segment_text):
                    continue

            segments.append({"text": cleaned_segment_text, "type": segment_type})

        last_end = end_index

    if last_end < len(text_content):
        remaining_text = text_content[last_end:]
        cleaned_remaining_text = robust_clean_text(remaining_text)
        if cleaned_remaining_text:
            segments.append({"text": cleaned_remaining_text, "type": "paragraph"})

    return segments

In [ ]:
# ext_science = []
# ds_wiki = load_dataset("wiki40b", "ru", split="train", streaming=True)
# count = 0
# target_count = WIKI_LIMIT

# for item in tqdm(ds_wiki, total=target_count, desc="Wiki Stream"):
#     if count >= target_count: break

#     raw_text = item['text']
#     clean_text = fix_wiki_encoding(raw_text)
#     if len(clean_text) > 100 and "_START_" not in clean_text:
#         ext_science.append(clean_text)
#         count += 1

# print(f"Loaded {len(ext_science)} clean Wiki samples.")
# if len(ext_science) > 0:
#     print("Sample check:", ext_science[0][:100])


In [ ]:
ext_science = []
ds_wiki = load_dataset("wiki40b", "ru", split="train", streaming=True)
count = 0
target_count = WIKI_LIMIT

for item in tqdm(ds_wiki, total=target_count, desc="Wiki Stream"):
    if count >= target_count: break

    raw_text = item['text']
    # Use fix_wiki_encoding first to handle the raw text.
    cleaned_full_text = fix_wiki_encoding(raw_text)

    # Then parse into segments
    segments = parse_wiki_segments(cleaned_full_text)

    for segment in segments:
        text = segment['text']
        # Filter for length and append
        if len(text) > 100:
            ext_science.append(text)
            count += 1
            if count >= target_count: # Check limit after each append
                break


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

Wiki Stream:  47%|████▋     | 94129/200000 [02:12<02:29, 708.88it/s] 


# Merging datasets (by genres)

In [ ]:
# get Syntagrus genre tag
def get_syn_texts(genres):
    return df_syn[df_syn['genre'].isin(genres)]['text'].tolist()


In [ ]:
journ_texts = get_syn_texts(['journalism', 'news', 'publicistic']) + ext_journalism
print(f"Domain 'Journalism': {len(journ_texts)} samples")

fiction_texts = get_syn_texts(['fiction', 'literature']) + ext_fiction
print(f"Domain 'Fiction':    {len(fiction_texts)} samples")

science_texts = get_syn_texts(['nonfiction', 'wiki', 'science']) + ext_science
print(f"Domain 'Science':    {len(science_texts)} samples")


Domain 'Journalism': 53 samples
Domain 'Fiction':    17 samples
Domain 'Science':    200023 samples


# Generating jsons by genres

In [ ]:
domains = {
    "journalism": journ_texts,
    "fiction": fiction_texts,
    "science": science_texts
}

all_train_texts = []
all_test_texts = []

for domain_name, texts in domains.items():
    train_txt, test_txt = train_test_split(texts, test_size=0.1, random_state=42)

    create_json_dataset(train_txt, f"train_{domain_name}.json")
    create_json_dataset(test_txt, f"test_{domain_name}.json")

    all_train_texts.extend(train_txt)
    all_test_texts.extend(test_txt)


Processing 47 texts for train_journalism.json...


100%|██████████| 47/47 [00:00<00:00, 158.76it/s]


Saved /content/drive/MyDrive/omg_diploma_2025/Diploma_Data/train_journalism.json
Processing 6 texts for test_journalism.json...


100%|██████████| 6/6 [00:00<00:00, 178.65it/s]


Saved /content/drive/MyDrive/omg_diploma_2025/Diploma_Data/test_journalism.json
Processing 15 texts for train_fiction.json...


...

Saved /content/drive/MyDrive/omg_diploma_2025/Diploma_Data/test_fiction.json
Processing 180020 texts for train_science.json...


100%|██████████| 180020/180020 [01:38<00:00, 1834.12it/s]


Saved /content/drive/MyDrive/omg_diploma_2025/Diploma_Data/train_science.json
Processing 20003 texts for test_science.json...


100%|██████████| 20003/20003 [00:11<00:00, 1704.57it/s]


Saved /content/drive/MyDrive/omg_diploma_2025/Diploma_Data/test_science.json


In [ ]:
import random
random.shuffle(all_train_texts)

print("\nProcessing Combined 'ALL' Dataset...")
create_json_dataset(all_train_texts, "train_all.json")
create_json_dataset(all_test_texts, "test_all.json")

print("\nDONE!datasets ready for BERT:")
print("1. train/test_all.json        (Generalist Model)")
print("2. train/test_journalism.json (Specialist Experiment)")
print("3. train/test_fiction.json    (Specialist Experiment)")
print("4. train/test_science.json    (Specialist Experiment)")



Processing Combined 'ALL' Dataset...
Processing 180082 texts for train_all.json...


100%|██████████| 180082/180082 [01:29<00:00, 2005.86it/s]


Saved /content/drive/MyDrive/omg_diploma_2025/Diploma_Data/train_all.json
Processing 20011 texts for test_all.json...


100%|██████████| 20011/20011 [00:12<00:00, 1607.51it/s]


Saved /content/drive/MyDrive/omg_diploma_2025/Diploma_Data/test_all.json

DONE!datasets ready for BERT:
1. train/test_all.json        (Generalist Model)
2. train/test_journalism.json (Specialist Experiment)
3. train/test_fiction.json    (Specialist Experiment)
4. train/test_science.json    (Specialist Experiment)


# Testing data

In [ ]:
syntagrus_test_url = ["https://raw.githubusercontent.com/UniversalDependencies/UD_Russian-SynTagRus/master/ru_syntagrus-ud-test.conllu"]
df_syn_test = process_syntagrus_files(syntagrus_test_url, "SynTagRus TEST")

--- PROCESSING SynTagRus TEST ---


8800it [00:04, 1802.32it/s]


## *domains testing*

In [ ]:
def get_texts_by_genre(df, genres):
    return df[df['genre'].isin(genres)]['text'].tolist()

print("\n--- ASSEMBLING FINAL DATASETS ---")

domains = {
    "journalism": {
        "syn_keywords": ['journalism', 'news', 'publicistic'],
        "external": ext_journalism
    },
    "fiction": {
        "syn_keywords": ['fiction', 'literature'],
        "external": ext_fiction
    },
    "science": {
        "syn_keywords": ['nonfiction', 'wiki', 'science'],
        "external": ext_science
    }
}


--- ASSEMBLING FINAL DATASETS ---


In [ ]:
all_train_texts = []

for domain_name, sources in domains.items():
    syn_train_txt = get_texts_by_genre(df_syn, sources['syn_keywords'])
    domain_train = syn_train_txt + sources['external']

    create_json_dataset(domain_train, f"train_{domain_name}.json")
    all_train_texts.extend(domain_train)

    bench_test_txt = get_texts_by_genre(df_syn_test, sources['syn_keywords'])

    if len(bench_test_txt) == 0:
        print(f"No test data found for {domain_name} in SynTagRus Test!")
    else:
        create_json_dataset(bench_test_txt, f"bench_test_{domain_name}.json")

Processing 53 texts for train_journalism.json...


100%|██████████| 53/53 [00:00<00:00, 134.89it/s]


Saved /content/drive/MyDrive/omg_diploma_2025/Diploma_Data/train_journalism.json
Processing 4 texts for bench_test_journalism.json...


100%|██████████| 4/4 [00:00<00:00, 172.26it/s]


Saved /content/drive/MyDrive/omg_diploma_2025/Diploma_Data/bench_test_journalism.json
Processing 17 texts for train_fiction.json...


...

Saved /content/drive/MyDrive/omg_diploma_2025/Diploma_Data/bench_test_fiction.json
Processing 200023 texts for train_science.json...


100%|██████████| 200023/200023 [01:35<00:00, 2084.11it/s]


Saved /content/drive/MyDrive/omg_diploma_2025/Diploma_Data/train_science.json
Processing 3 texts for bench_test_science.json...


100%|██████████| 3/3 [00:00<00:00, 51.84it/s]


Saved /content/drive/MyDrive/omg_diploma_2025/Diploma_Data/bench_test_science.json


## *merged domain training & testing*

In [ ]:
random.shuffle(all_train_texts)

# Create Validation Set (Just 5% of training data to check convergence)
train_txt, val_txt = train_test_split(all_train_texts, test_size=0.05, random_state=42)

print("\nProcessing Generalist Datasets...")
create_json_dataset(train_txt, "train_all.json")
create_json_dataset(val_txt, "val_internal.json")

print("\nDONE! Dataset Manifest:")
print("1. train_all.json         -> Main Training Data (Gazeta+Opus+Wiki+SynTagRusTrain)")
print("2. val_internal.json      -> For checking loss during training")
print("3. train_[genre].json     -> For Domain Adaptation Training")
print("4. bench_test_[genre].json -> OFFICIAL BENCHMARK (SynTagRus Test Only)")


Processing Generalist Datasets...
Processing 190088 texts for train_all.json...


100%|██████████| 190088/190088 [01:44<00:00, 1811.54it/s]


Saved /content/drive/MyDrive/omg_diploma_2025/Diploma_Data/train_all.json
Processing 10005 texts for val_internal.json...


100%|██████████| 10005/10005 [00:03<00:00, 2680.31it/s]


Saved /content/drive/MyDrive/omg_diploma_2025/Diploma_Data/val_internal.json

DONE! Dataset Manifest:
1. train_all.json         -> Main Training Data (Gazeta+Opus+Wiki+SynTagRusTrain)
2. val_internal.json      -> For checking loss during training
3. train_[genre].json     -> For Domain Adaptation Training
4. bench_test_[genre].json -> OFFICIAL BENCHMARK (SynTagRus Test Only)


## Visuzlization

In [ ]:
data_files = {
    "train": os.path.join(output_dir, "train_all.json"),
    "validation": os.path.join(output_dir, "val_internal.json")
}
raw_datasets = load_dataset("json", data_files=data_files, streaming=True)


In [ ]:
import pandas as pd

def visualize_dataset(dataset_split, num_samples=5, id2label=ID_TO_LABEL):
    samples = []
    # Iterate over the streaming dataset to collect samples
    for i, example in enumerate(dataset_split):
        if i >= num_samples:
            break
        samples.append(example)

    # If no samples were collected (e.g., empty dataset or num_samples=0)
    if not samples:
        return pd.DataFrame()

    df = pd.DataFrame(samples)
    if id2label:
        def get_label_names(tag_ids):
            return [id2label[int(t)] for t in tag_ids]

        df['ner_tags_names'] = df['ner_tags'].apply(get_label_names)
        cols = ['tokens', 'ner_tags_names', 'ner_tags'] + [c for c in df.columns if c not in ['tokens', 'ner_tags_names', 'ner_tags']]
        df = df[cols]
    pd.set_option('display.max_colwidth', None)

    return df

df_view = visualize_dataset(raw_datasets['train'], num_samples=3, id2label=ID_TO_LABEL)

display(df_view)


,tokens,ner_tags_names,ner_tags
0,"[Гетероду, плекс, гибридный, участок, молекул, ДНК, вступивших, в, рекомбинацию, состоящий, из, двух, цепей, от, каждой, из, рекомбинирующих, ДНК, (то, есть, разного, происхождения), Одиночные, цепи, удерживаются, друг, с, другом, благодаря, водородным, связям, между, комплементарными, основаниями, Участки, в, которых, основания, не, подходят, друг, другу, приобретают, форму, петли, Чем, больше, в, гетеродуплексе, неспаренных, участков, тем, менее, он, устойчив, поэтому, может, осуществляться, подгонка, оснований, друг, под, друга, В, некоторых, случаях, гетеродуплекс, могут, образовывать, одиночные, цепи, РНК, Чтобы, определить, содержит, ли, ДНК, интроны, можно, провести, её, транскрипцию, с, последующей, гибридизацией, полученной, цепи, РНК, с, матрицей, исходной, ДНК, Если, ДНК, содержит, интроны, то, в, транскрибированной, с, ...]","[O, -, O, O, O, ,, O, O, ,, O, O, O, O, O, O, O, O, O, O, O, O, ., O, O, O, O, O, O, O, O, O, O, O, ., ,, O, O, O, O, O, O, ,, O, O, ., O, O, O, O, O, ,, O, O, O, ,, O, O, O, O, O, O, O, ., O, O, O, O, O, O, O, O, ., O, ,, O, O, O, ,, O, O, O, O, O, O, O, O, O, O, O, -, O, ., O, O, O, ,, O, O, O, O, ...]","[0, 7, 0, 0, 0, 2, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 0, 0, 0, 0, 0, 0, 2, 0, 0, 1, 0, 0, 0, 0, 0, 2, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 1, 0, 0, 0, 2, 0, 0, 0, 0, ...]"
1,"[Родился, 23, декабря, 1923, года, в, армянской, семье, в, городе, Ленинакан, (ныне, г, Гюмри, Армения), ССР, Армении, в, многодетной, семье, плотника, вагонного, депо, на, железной, дороге, Апета, Аматуни, Назван, был, Ашотом, в, честь, друга, отца, рано, ушедшего, из, жизни, В, семье, помимо, него, было, ещё, 11, детей, каждого, из, которых, отец, приучил, к, ответственности, и, труду, позже, Ашот, Аматуни, говорил, что, именно, благодаря, этим, чертам, привитым, отцом, он, с, отличием, окончил, школу, и, поступил, в, Ленинаканский, педагогический, институт, С, 1930, по, 1940, годы, учился, в, 51-й, железнодорожной, средней, школе, Ленинакана, Семья]","[O, O, O, O, O, O, O, O, O, O, O, O, ., ,, O, O, O, O, O, O, O, O, O, O, O, O, O, ., O, O, O, O, O, O, ,, O, O, O, ., O, O, O, O, O, O, O, ,, O, O, O, O, O, O, O, O, ,, O, O, O, ,, O, O, O, O, ,, O, ,, O, O, O, O, O, O, O, O, O, O, ., O, O, O, O, O, O, O, O, O, O, O, ., O]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 2, 0, 0, 0, 0, 2, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]"
2,"[Окончил, филологический, факультет, Новгородского, государственного, университета, получив, специальность, учителя, литературы, В, 2002, году, опубликован, сборник, стихотворений, Бударагина, Звёзды, в, квадрате, окна, В, октябре, 2003, года, принял, участие, в, Третьем, форуме, молодых, писателей, России, (мастер-класс, Кирилла, Ковальджи), Как, литератор, публиковался, в, калининградском, журнале, Насекомое, До, 2007-го, года, работал, главным, редактором, официального, сайта, партии, Единая, Россия, был, членом, Федерального, политсовета, Молодой, гвардии, Единой, России, Писал, материалы, для, многих, изданий, Известия, (на, сайте, Известий, размещены, его, материалы, за, 2006-2008, годы), Русский, Журнал, (размещены, его, публикации, за, 2005-2008, годы), GlobalRus, ru, (размещены, его, статьи, за, 2005-2007, годы), а, также, региональных, изданий, Основатель, Новгородского, политического, ...]","[O, O, O, O, O, ,, O, O, O, ., O, O, O, O, O, O, "", O, O, O, ""., O, O, O, O, O, O, O, O, O, O, O, O, O, O, ., O, ,, O, O, O, "", ""., O, O, O, O, O, O, O, O, "", O, "",, O, O, O, "", O, O, O, ""., O, O, O, O, - "", "" , O, "", "" , O, O, O, O, O, , "", O, "" , O, O, O, O, O, , "", ., "" , O, O, O, O, O, ,, O, O, O, ., "", O, O, ...]","[0, 0, 0, 0, 0, 2, 0, 0, 

In [ ]:
def audit_example(dataset_split, index, id2label):
    example = dataset_split[index]
    tokens = example['tokens']
    tags = example['ner_tags']

    data = []
    for t, tag_id in zip(tokens, tags):
        data.append({
            "Token": t,
            "Tag ID": tag_id,
            "Label": id2label[int(tag_id)]
        })

    return pd.DataFrame(data)

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', None)

def audit_example_from_dict(example_dict, id2label):
    """
    Audits a single example given as a dictionary.
    """
    tokens = example_dict['tokens']
    tags = example_dict['ner_tags']

    data = []
    for t, tag_id in zip(tokens, tags):
        data.append({
            "Token": t,
            "Tag ID": tag_id,
            "Label": id2label[int(tag_id)]
        })
    return pd.DataFrame(data)

def audit_multiple_examples(dataset_split, indices, id2label):
    collected_examples = {}
    max_index = max(indices) if indices else -1

    for i, example in enumerate(dataset_split):
        if i > max_index and len(collected_examples) == len(indices):
            break
        if i in indices:
            collected_examples[i] = example

    for idx in indices:
        if idx in collected_examples:
            print(f"\n example index = {idx}")
            df = audit_example_from_dict(collected_examples[idx], id2label)
            display(df)
        else:
            print(f"\n Example at index {idx} not found. Dataset might be smaller than requested index or not enough items were streamed.")

audit_multiple_examples(raw_datasets['train'], [0, 1, 2, 100], ID_TO_LABEL)



 example index = 0


,Token,Tag ID,Label
0,Гетероду,0,O
1,плекс,7,-
2,гибридный,0,O
3,участок,0,O
4,молекул,0,O
5,ДНК,2,","
6,вступивших,0,O
7,в,0,O
8,рекомбинацию,2,","
9,состоящий,0,O



 example index = 1


,Token,Tag ID,Label
0,Родился,0,O
1,23,0,O
2,декабря,0,O
3,1923,0,O
4,года,0,O
5,в,0,O
6,армянской,0,O
7,семье,0,O
8,в,0,O
9,городе,0,O



 example index = 2


,Token,Tag ID,Label
0,Окончил,0,O
1,филологический,0,O
2,факультет,0,O
3,Новгородского,0,O
4,государственного,0,O
5,университета,2,","
6,получив,0,O
7,специальность,0,O
8,учителя,0,O
9,литературы,1,.



 example index = 100


,Token,Tag ID,Label
0,В,0,O
1,декабре,0,O
2,2017,0,O
3,российские,0,O
4,и,0,O
5,сирийские,0,O
6,СМИ,0,O
7,сообщили,2,","
8,что,0,O
9,в,0,O
